# ViCAM - Viral fine-tuning of Cambriam

In [ ]:
from Bio import SeqIO


In [5]:
#input_fasta = "../data/raw/URVDBv29-prot_clustered.fasta"
input_fasta = "../data/processed/C-RVDBv29_no_poly/train.fasta"

records = list(SeqIO.parse(input_fasta, "fasta"))

#records = [record for record in records if 'poly' in record.description]

In [6]:
len(records)

568485

In [7]:
mn=0
mx=0
for record in records:
    if len(record.seq) < mn or mn == 0:
        mn = len(record.seq)
    if len(record.seq) > mx:
        mx = len(record.seq)
print(f"Min length: {mn}")
print(f"Max length: {mx}")


Min length: 11
Max length: 2047


In [3]:
count = 0
while count < 10:
    record = records[count]
    print(record.description)
    count += 1

    

acc|GENBANK|UJJ65121.1|GENBANK|OM336640|surface glycoprotein [Severe acute respiratory syndrome coronavirus 2]
acc|GENBANK|UJJ65122.1|GENBANK|OM336640|ORF3a protein [Severe acute respiratory syndrome coronavirus 2]
acc|GENBANK|UJJ65123.1|GENBANK|OM336640|envelope protein [Severe acute respiratory syndrome coronavirus 2]
acc|GENBANK|UJJ65124.1|GENBANK|OM336640|membrane glycoprotein [Severe acute respiratory syndrome coronavirus 2]
acc|GENBANK|UJJ65125.1|GENBANK|OM336640|ORF6 protein [Severe acute respiratory syndrome coronavirus 2]
acc|GENBANK|UJJ65126.1|GENBANK|OM336640|ORF7a protein [Severe acute respiratory syndrome coronavirus 2]
acc|GENBANK|UJJ65127.1|GENBANK|OM336640|ORF7b [Severe acute respiratory syndrome coronavirus 2]
acc|GENBANK|UJJ65128.1|GENBANK|OM336640|ORF8 protein [Severe acute respiratory syndrome coronavirus 2]
acc|GENBANK|UJJ65129.1|GENBANK|OM336640|nucleocapsid phosphoprotein [Severe acute respiratory syndrome coronavirus 2]
acc|GENBANK|UJJ65130.1|GENBANK|OM336640|OR

# inference

In [13]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "2"
import torch
from esm.models.esmc import ESMC
from esm.tokenization import get_esmc_model_tokenizers

tokenizer = get_esmc_model_tokenizers()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [ ]:
model_path= '/stor/work/Wilke/wilkelab/pLMs_checkpoints/ESMC/esmc_300m_2024_12_v0.pth'
state_dict = torch.load(model_path, map_location=device, weights_only=True)
esmc300m = ESMC(d_model=960, n_heads=15, n_layers=30, tokenizer=get_esmc_model_tokenizers())
esmc300m.load_state_dict(state_dict)
esmc300m.eval()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
esmc300m.to(device)
esmc300m

ESMC(
  (embed): Embedding(64, 960)
  (transformer): TransformerStack(
    (blocks): ModuleList(
      (0-29): 30 x UnifiedTransformerBlock(
        (attn): MultiHeadAttention(
          (layernorm_qkv): Sequential(
            (0): LayerNorm((960,), eps=1e-05, elementwise_affine=True)
            (1): Linear(in_features=960, out_features=2880, bias=False)
          )
          (out_proj): Linear(in_features=960, out_features=960, bias=False)
          (q_ln): LayerNorm((960,), eps=1e-05, elementwise_affine=True)
          (k_ln): LayerNorm((960,), eps=1e-05, elementwise_affine=True)
          (rotary): RotaryEmbedding()
        )
        (ffn): Sequential(
          (0): LayerNorm((960,), eps=1e-05, elementwise_affine=True)
          (1): Linear(in_features=960, out_features=5120, bias=False)
          (2): SwiGLU()
          (3): Linear(in_features=2560, out_features=960, bias=False)
        )
      )
    )
    (norm): LayerNorm((960,), eps=1e-05, elementwise_affine=True)
  )
  (sequ

In [18]:
tokens = tokenizer(["AAAAAAA", "AAAAA"], return_tensors="pt", padding=True)
tokens.to(device) 

{'input_ids': tensor([[0, 5, 5, 5, 5, 5, 5, 5, 2],
        [0, 5, 5, 5, 5, 5, 2, 1, 1]], device='cuda:0'), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 1, 1, 0, 0]], device='cuda:0')}

In [19]:
esmc300m(tokens["input_ids"])

ESMCOutput(sequence_logits=tensor([[[-38.0333, -38.0360, -38.0323,  ..., -38.0285, -38.0664, -38.0540],
         [-40.2790, -40.2907, -40.3479,  ..., -40.3142, -40.3002, -40.3140],
         [-36.3614, -36.3531, -36.3800,  ..., -36.4196, -36.3916, -36.4265],
         ...,
         [-36.4638, -36.4345, -36.5000,  ..., -36.5169, -36.4898, -36.5214],
         [-35.7786, -35.7601, -35.8030,  ..., -35.8089, -35.8304, -35.8100],
         [-33.7242, -33.7013, -33.7387,  ..., -33.7467, -33.7438, -33.7541]],

        [[-38.0487, -38.0396, -38.0528,  ..., -38.0591, -38.0906, -38.0756],
         [-40.0131, -40.0105, -40.0686,  ..., -40.0690, -40.0437, -40.0478],
         [-36.1433, -36.1190, -36.1505,  ..., -36.2211, -36.2072, -36.1873],
         ...,
         [-32.9298, -32.9026, -32.9321,  ..., -32.9482, -32.9679, -32.9473],
         [-19.2742, -19.4478, -19.2775,  ..., -19.3356, -19.1872, -19.4463],
         [-19.2742, -19.4478, -19.2775,  ..., -19.3356, -19.1872, -19.4463]]],
       device='cu

In [ ]:
path_checkpoint = "/stor/work/Wilke/luiz/ViCAM/checkpoints/ViCAM_300M/v03_no_ploy/epoch=1-val_loss=1.73.ckpt"
state_dict = torch.load(path_checkpoint)["state_dict"]
new_state_dict = {k.replace("model.", ""): v for k, v in state_dict.items()}
vicam = ESMC(d_model=960, n_heads=15, n_layers=30, tokenizer=get_esmc_model_tokenizers())
vicam.load_state_dict(new_state_dict)
vicam.eval()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
vicam.to(device)
vicam


ESMC(
  (embed): Embedding(64, 960)
  (transformer): TransformerStack(
    (blocks): ModuleList(
      (0-29): 30 x UnifiedTransformerBlock(
        (attn): MultiHeadAttention(
          (layernorm_qkv): Sequential(
            (0): LayerNorm((960,), eps=1e-05, elementwise_affine=True)
            (1): Linear(in_features=960, out_features=2880, bias=False)
          )
          (out_proj): Linear(in_features=960, out_features=960, bias=False)
          (q_ln): LayerNorm((960,), eps=1e-05, elementwise_affine=True)
          (k_ln): LayerNorm((960,), eps=1e-05, elementwise_affine=True)
          (rotary): RotaryEmbedding()
        )
        (ffn): Sequential(
          (0): LayerNorm((960,), eps=1e-05, elementwise_affine=True)
          (1): Linear(in_features=960, out_features=5120, bias=False)
          (2): SwiGLU()
          (3): Linear(in_features=2560, out_features=960, bias=False)
        )
      )
    )
    (norm): LayerNorm((960,), eps=1e-05, elementwise_affine=True)
  )
  (sequ